# 00 — Data Preparation
> Run this notebook first before any model notebook.
>
> Outputs: `prepared_classification.pkl`, `prepared_forecasting.pkl`, `scaler.pkl`, `encoder.pkl`, `encoder_reason.pkl`


In [1]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('✅ Imports complete.')


✅ Imports complete.


In [2]:
# FILE PATH — update this to your local path
DATA_FILE = r'D:\Courses\IIT\AIDS\FYP\Preventing-Mechanism\Datasets\MBP_50k_Strong.xlsx'

TIME_STEPS = 20
H = 15

STATE_COL = 'Machine States'

KNOWN_BREAKDOWNS = [
    'High Thread Tension',
    'High foot pressure',
    'Running with no thread',
]
ALLOWED_STATES = ['Healthy'] + KNOWN_BREAKDOWNS

VIB_GROUP_1 = [f'{i}-{i+10}Hz' for i in range(10,  100, 10)]
VIB_GROUP_2 = [f'{i}-{i+10}Hz' for i in range(100, 300, 10)]
VIB_GROUP_3 = [f'{i}-{i+10}Hz' for i in range(300, 610, 10)]
ELEC_FEATS  = ['machineVoltageMean','machineVoltageMin','machineVoltageMax',
               'machineCurrentMean','machineCurrentMin','machineCurrentMax']

print(f'✅ Config loaded.')
print(f'   TIME_STEPS : {TIME_STEPS}')
print(f'   H          : {H}')
print(f'   Known breakdowns: {KNOWN_BREAKDOWNS}')


✅ Config loaded.
   TIME_STEPS : 20
   H          : 15
   Known breakdowns: ['High Thread Tension', 'High foot pressure', 'Running with no thread']


In [3]:
# LOAD DATA
master_df = pd.read_excel(DATA_FILE)

# Filter valid vibration records (must start with '10')
master_df = master_df[master_df['machineVibration'].astype(str).str.startswith('10')].copy()

# Fill empty Machine States as 'Healthy'
master_df[STATE_COL] = master_df[STATE_COL].fillna('Healthy')

# Keep only allowed states
master_df = master_df[master_df[STATE_COL].isin(ALLOWED_STATES)].reset_index(drop=True)

# Sort chronologically
master_df['dateTime'] = pd.to_datetime(master_df['dateTime'])
master_df = master_df.sort_values('dateTime').reset_index(drop=True)

# time_gap: seconds between consecutive pedal presses (capped at 300s)
master_df['time_gap'] = master_df['dateTime'].diff().dt.total_seconds().fillna(0).clip(upper=300)

print(f'✅ Data loaded: {len(master_df)} total records')
print(f'   Date range: {master_df["dateTime"].iloc[0]} to {master_df["dateTime"].iloc[-1]}')
print(f'\nClass distribution:')
print(master_df[STATE_COL].value_counts())
print(f'\nHealthy %   : {(master_df[STATE_COL]=="Healthy").mean()*100:.1f}%')
print(f'Breakdown % : {(master_df[STATE_COL]!="Healthy").mean()*100:.1f}%')


✅ Data loaded: 50000 total records
   Date range: 2025-01-01 00:00:04.186000 to 2025-01-02 15:54:43.657000

Class distribution:
Machine States
Healthy                   47095
Running with no thread      993
High Thread Tension         973
High foot pressure          939
Name: count, dtype: int64

Healthy %   : 94.2%
Breakdown % : 5.8%


In [4]:
# FEATURE EXTRACTION (67 features: 60 vib + 6 elec + 1 time_gap)
def extract_features(df):
    vib_records = []
    for val in df['machineVibration']:
        vib_dict = {}
        parts = str(val).split(',')
        try:
            for i in range(0, len(parts)-1, 2):
                f_start = int(parts[i])
                vib_dict[f'{f_start}-{f_start+10}Hz'] = int(parts[i+1])
        except: pass
        vib_records.append(vib_dict)

    expected_vib = [f'{i}-{i+10}Hz' for i in range(10, 610, 10)]
    vib_df      = pd.DataFrame(vib_records).reindex(columns=expected_vib, fill_value=0)
    elec_df     = df[ELEC_FEATS].ffill().fillna(0).reset_index(drop=True)
    time_gap_df = df[['time_gap']].reset_index(drop=True)
    return pd.concat([vib_df.reset_index(drop=True), elec_df, time_gap_df], axis=1)

X_all = extract_features(master_df)
y_all = master_df[STATE_COL].values
print(f'✅ Features extracted: {X_all.shape}  (60 vib + 6 elec + 1 time_gap = 67 features)')


✅ Features extracted: (50000, 67)  (60 vib + 6 elec + 1 time_gap = 67 features)


---
## Classification Data (FAG-TFT + Anomaly Gate)


In [5]:
# ENCODE LABELS FOR CLASSIFICATION
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y_all)
num_classes = len(encoder.classes_)
print(f'✅ Classes: {list(encoder.classes_)}')

# TIME-BASED SPLIT — correct for time series
split_idx = int(len(X_all) * 0.80)
X_train_raw = X_all.values[:split_idx]
X_test_raw  = X_all.values[split_idx:]
y_train     = y_encoded[:split_idx]
y_test      = y_encoded[split_idx:]

# SCALE — fit on train only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled  = scaler.transform(X_test_raw)

print(f'   Train: {X_train_scaled.shape}  Test: {X_test_scaled.shape}')
print(f'   Train class counts: {np.bincount(y_train)}')
print(f'   Test  class counts: {np.bincount(y_test)}')


✅ Classes: ['Healthy', 'High Thread Tension', 'High foot pressure', 'Running with no thread']
   Train: (40000, 67)  Test: (10000, 67)
   Train class counts: [37678   776   770   776]
   Test  class counts: [9417  197  169  217]


In [6]:
# SEQUENCE CREATION FOR CLASSIFICATION
def create_sequences(X, y, time_steps):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:i+time_steps])
        ys.append(y[i+time_steps])
    return np.array(Xs), np.array(ys)

X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train, TIME_STEPS)
X_test_seq,  y_test_seq  = create_sequences(X_test_scaled,  y_test,  TIME_STEPS)

print(f'✅ Sequences created (before balancing).')
print(f'   Train class counts: {np.bincount(y_train_seq)}')

# BALANCE — undersample Healthy to 3x minority class
healthy_cls_idx = list(encoder.classes_).index('Healthy')
counts          = np.bincount(y_train_seq)
minority_count  = min(counts[j] for j in range(len(counts)) if j != healthy_cls_idx and counts[j] > 0)
target_healthy  = minority_count * 3

idx_healthy  = np.where(y_train_seq == healthy_cls_idx)[0]
idx_others   = np.where(y_train_seq != healthy_cls_idx)[0]
idx_healthy_sampled = np.random.choice(idx_healthy, size=min(target_healthy, len(idx_healthy)), replace=False)
idx_balanced = np.concatenate([idx_healthy_sampled, idx_others])
np.random.shuffle(idx_balanced)

X_train_seq = X_train_seq[idx_balanced]
y_train_seq = y_train_seq[idx_balanced]

print(f'✅ Classification sequences balanced.')
print(f'   Train: {X_train_seq.shape}')
print(f'   Test : {X_test_seq.shape}')
print(f'   Train class counts (balanced): {np.bincount(y_train_seq)}')
print(f'   Test  class counts           : {np.bincount(y_test_seq)}')


✅ Sequences created (before balancing).
   Train class counts: [37658   776   770   776]
✅ Classification sequences balanced.
   Train: (4632, 20, 67)
   Test : (9980, 20, 67)
   Train class counts (balanced): [2310  776  770  776]
   Test  class counts           : [9397  197  169  217]


---
## Forecasting Data (TFT Forecaster + LSTM/GRU Baselines)


In [7]:
# CREATE FORECASTING LABELS (vectorised for speed on large datasets)
master_df['breakdown_now'] = (master_df[STATE_COL] != 'Healthy').astype(int)
is_breakdown = (master_df[STATE_COL] != 'Healthy').values
states       = master_df[STATE_COL].values
n            = len(master_df)

future_breakdown = np.zeros(n, dtype=int)
future_reason    = np.array(['No Breakdown'] * n, dtype=object)

for i in range(n):
    window_end = min(i + 1 + H, n)
    for j in range(i + 1, window_end):
        if is_breakdown[j]:
            future_breakdown[i] = 1
            future_reason[i]    = states[j]
            break

master_df['future_breakdown'] = future_breakdown
master_df['future_reason']    = future_reason

# Remove rows where breakdown is already happening
df_forecast = master_df[master_df['breakdown_now'] == 0].copy().reset_index(drop=True)

print(f'✅ Forecasting labels created.')
print(f'   Total records (active breakdowns removed): {len(df_forecast)}')
print(f'   Safe (no breakdown coming)  : {(df_forecast["future_breakdown"]==0).sum()} ({(df_forecast["future_breakdown"]==0).mean()*100:.1f}%)')
print(f'   Approaching breakdown       : {(df_forecast["future_breakdown"]==1).sum()} ({(df_forecast["future_breakdown"]==1).mean()*100:.1f}%)')
print(f'\n   Approaching breakdown by type:')
print(df_forecast[df_forecast['future_breakdown']==1]['future_reason'].value_counts().to_string())


✅ Forecasting labels created.
   Total records (active breakdowns removed): 47095
   Safe (no breakdown coming)  : 44080 (93.6%)
   Approaching breakdown       : 3015 (6.4%)

   Approaching breakdown by type:
future_reason
High Thread Tension       1005
High foot pressure        1005
Running with no thread    1005


In [8]:
# PREPARE FORECASTING SEQUENCES

# BALANCE — undersample Safe to 4x Breakdown count
df_safe      = df_forecast[df_forecast['future_breakdown'] == 0]
df_breakdown = df_forecast[df_forecast['future_breakdown'] == 1]
n_safe_target = min(len(df_breakdown) * 4, len(df_safe))
df_safe_sampled = df_safe.sample(n=n_safe_target, random_state=123)
df_forecast_bal = pd.concat([df_safe_sampled, df_breakdown]).sort_values('dateTime').reset_index(drop=True)

print(f'✅ Forecasting data balanced:')
print(f'   Safe             : {(df_forecast_bal["future_breakdown"]==0).sum()}')
print(f'   Breakdown Coming : {(df_forecast_bal["future_breakdown"]==1).sum()}')

X_forecast        = extract_features(df_forecast_bal)
y_forecast_binary = df_forecast_bal['future_breakdown'].values

encoder_reason    = LabelEncoder()
y_forecast_reason = encoder_reason.fit_transform(df_forecast_bal['future_reason'].values)
num_reason_classes = len(encoder_reason.classes_)
print(f'   Reason classes: {list(encoder_reason.classes_)}')

# TIME-BASED SPLIT — no sequence overlap
split_idx = int(len(X_forecast) * 0.80)
X_fc_train_raw  = X_forecast.values[:split_idx]
X_fc_test_raw   = X_forecast.values[split_idx:]
y_fc_train_bin  = y_forecast_binary[:split_idx]
y_fc_test_bin   = y_forecast_binary[split_idx:]
y_fc_train_type = y_forecast_reason[:split_idx]
y_fc_test_type  = y_forecast_reason[split_idx:]

print(f'   Train binary counts: {np.bincount(y_fc_train_bin)}')
print(f'   Test  binary counts: {np.bincount(y_fc_test_bin)}')

X_fc_train_scaled = scaler.transform(X_fc_train_raw)
X_fc_test_scaled  = scaler.transform(X_fc_test_raw)

X_fc_train_seq, y_fc_train_bin_seq  = create_sequences(X_fc_train_scaled, y_fc_train_bin,  TIME_STEPS)
X_fc_test_seq,  y_fc_test_bin_seq   = create_sequences(X_fc_test_scaled,  y_fc_test_bin,   TIME_STEPS)
_, y_fc_train_type_seq              = create_sequences(X_fc_train_scaled, y_fc_train_type, TIME_STEPS)
_, y_fc_test_type_seq               = create_sequences(X_fc_test_scaled,  y_fc_test_type,  TIME_STEPS)

print(f'✅ Forecasting sequences created.')
print(f'   Train: {X_fc_train_seq.shape}  Test: {X_fc_test_seq.shape}')
print(f'   Binary class counts (train): {np.bincount(y_fc_train_bin_seq)}')
print(f'   Binary class counts (test) : {np.bincount(y_fc_test_bin_seq)}')


✅ Forecasting data balanced:
   Safe             : 12060
   Breakdown Coming : 3015
   Reason classes: ['High Thread Tension', 'High foot pressure', 'No Breakdown', 'Running with no thread']
   Train binary counts: [9642 2418]
   Test  binary counts: [2418  597]
✅ Forecasting sequences created.
   Train: (12040, 20, 67)  Test: (2995, 20, 67)
   Binary class counts (train): [9622 2418]
   Binary class counts (test) : [2410  585]


---
## Save All Artifacts


In [9]:
# SAVE ALL DATA
prepared_classification = {
    'X_train_seq'  : X_train_seq,
    'X_test_seq'   : X_test_seq,
    'y_train_seq'  : y_train_seq,
    'y_test_seq'   : y_test_seq,
    'num_classes'  : num_classes,
    'num_features' : X_train_seq.shape[2],
    'TIME_STEPS'   : TIME_STEPS,
    'VIB_GROUP_1'  : VIB_GROUP_1,
    'VIB_GROUP_2'  : VIB_GROUP_2,
    'VIB_GROUP_3'  : VIB_GROUP_3,
    'ELEC_FEATS'   : ELEC_FEATS,
    'feature_names': list(X_all.columns),
}

prepared_forecasting = {
    'X_fc_train_seq'      : X_fc_train_seq,
    'X_fc_test_seq'       : X_fc_test_seq,
    'y_fc_train_bin_seq'  : y_fc_train_bin_seq,
    'y_fc_test_bin_seq'   : y_fc_test_bin_seq,
    'y_fc_train_type_seq' : y_fc_train_type_seq,
    'y_fc_test_type_seq'  : y_fc_test_type_seq,
    'num_features'        : X_fc_train_seq.shape[2],
    'num_reason_classes'  : num_reason_classes,
    'TIME_STEPS'          : TIME_STEPS,
    'H'                   : H,
    'feature_names'       : list(X_forecast.columns),
}

with open('prepared_classification.pkl','wb') as f: pickle.dump(prepared_classification, f)
with open('prepared_forecasting.pkl','wb') as f:    pickle.dump(prepared_forecasting, f)
with open('scaler.pkl','wb') as f:                  pickle.dump(scaler, f)
with open('encoder.pkl','wb') as f:                 pickle.dump(encoder, f)
with open('encoder_reason.pkl','wb') as f:          pickle.dump(encoder_reason, f)

print('✅ All artifacts saved.')
print(f'   Classification : {num_classes} classes, {X_train_seq.shape[2]} features, TIME_STEPS={TIME_STEPS}')
print(f'   Forecasting    : binary + {num_reason_classes} reason classes, H={H}')


✅ All artifacts saved.
   Classification : 4 classes, 67 features, TIME_STEPS=20
   Forecasting    : binary + 4 reason classes, H=15
